# Synthetic Experiment Data · make data worth plotting

Edge systems are measured, not guessed. Before you can claim that one model is faster or one device runs cooler, you need a clean table of measurements from a repeatable experiment. The real measurements come from the labs later in the course. In this lab you generate realistic **synthetic** data instead, so you can practice the whole workflow now: design an experiment, decide how many runs it needs, record the numbers in a tidy format, and package it so anyone can reproduce it.

This is the groundwork for the next lab, where you turn this data into publication quality figures.

Work through it top to bottom. Run every code cell and read what comes back.

## How this notebook works

- **[Notebook cell]** runs here with **Shift+Enter**. A cell starting with `!` runs one shell command, and `%%bash` runs several lines.
- **[Terminal]** means open **File > New > Terminal** in JupyterLab and type it there. The environment work in Parts 6 to 8 is shown with `!` cells so you can stay in the notebook, but it is exactly what you would type in a terminal.

This lab uses `numpy` for the math and `pandas` for the tables. Both are already installed in the class image.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

### Preflight · check your environment

In [ ]:
preflight([
    check("python 3 available", commandOnPath("python3"),
          hint="python3 runs every cell in this notebook. It ships with the DGX Spark."),
    check("numpy importable", pythonImportable("numpy"),
          hint="numpy is preinstalled in the class image. Ask your instructor if this fails."),
    check("pandas importable", pythonImportable("pandas"),
          hint="pandas is preinstalled in the class image. Ask your instructor if this fails."),
    check("your home folder is writable", dirExists("~"),
          hint="You need a home folder to create the lab files."),
])

---
## Part 1 · Why an experiment must be reproducible

A result nobody can reproduce is not a result. Two habits make measured data trustworthy:

- **A fixed random seed** so the same code makes the same numbers every time.
- **A pinned environment** so the same package versions are used every time.

You will use both. First the seed, so your synthetic data is identical on every run and on every machine.

**[Notebook cell]** Set up a working folder and move into it, then create a single source of randomness. `numpy`'s `default_rng(seed)` returns a random number generator that always starts from the same place:

In [ ]:
from pathlib import Path
import numpy as np

(Path.home() / "experimentLab").mkdir(exist_ok=True)
%cd ~/experimentLab
labDir = Path.cwd()

SEED = 494
rng = np.random.default_rng(SEED)
print("workspace:", labDir)
print("two draws:", rng.random(2))

**Try it:** re-run the cell above. The two draws are identical every time, because seeding resets the generator to the same starting point. That is what makes an experiment repeatable.

---
## Part 2 · Model one measurement

Start with a single quantity: how long the device takes to run one inference, in milliseconds. Real latency is not a fixed number, it is a **distribution**: usually fast, occasionally slow, never negative. A **log-normal** distribution captures that shape well.

**[Notebook cell]** Draw 1000 latency samples with a realistic centre and a long right tail, then look at the shape with a few summary numbers:

In [ ]:
def sampleLatencyMs(rng, count, median=12.0, sigma=0.35):
    # log-normal: right-skewed, strictly positive, like real inference latency
    return np.exp(np.log(median) + sigma * rng.standard_normal(count))

latencies = sampleLatencyMs(rng, 1000)
print(f"n      = {len(latencies)}")
print(f"mean   = {latencies.mean():.2f} ms")
print(f"median = {np.median(latencies):.2f} ms")
print(f"p95    = {np.percentile(latencies, 95):.2f} ms")
print(f"p99    = {np.percentile(latencies, 99):.2f} ms")
print(f"max    = {latencies.max():.2f} ms")

Notice the mean sits above the median, and the p99 is far above both. That gap is the **tail**, the occasional slow inference. On edge devices the tail often matters more than the average, because it sets your worst-case response time.

---
## Part 3 · How many runs is enough?

"I ran it once and got 11 ms" is not an experiment. You repeat a measurement so the noise averages out, and you report how much it still varies. Two things decide how many runs you need.

**[Notebook cell]** First, **warm-up**. The first few runs are slow while caches fill and the GPU clocks up. Measure them and you bias the result, so you discard them. Here the first 5 runs are inflated and then dropped:

In [ ]:
raw = sampleLatencyMs(rng, 50)
raw[:5] *= 1.8       # the first runs are slow: cold caches, clock ramp
warm = raw[5:]       # discard the warm-up
print(f"with warm-up runs : mean {raw.mean():.2f} ms")
print(f"warm-up discarded : mean {warm.mean():.2f} ms")

**[Notebook cell]** Second, **how many**. Watch the estimate settle as you add runs. A few runs are noisy, and after enough the mean stops moving. The **standard error** tells you how much the estimate still wobbles:

In [ ]:
for n in [1, 3, 10, 30, 100, 300, 1000]:
    sample = sampleLatencyMs(rng, n)
    stderr = sample.std(ddof=1) / np.sqrt(n) if n > 1 else float("nan")
    print(f"n={n:4d}   mean={sample.mean():6.2f} ms   std-error={stderr:5.2f} ms")

The standard error shrinks with the square root of the number of runs, so going from 10 to 40 runs halves it, but you need 40 to 160 to halve it again. A practical rule for this course: **run at least 30 trials** after warm-up, and report the mean together with its spread. If the standard error is still large at 30, add more.

---
## Part 4 · A full experiment: several metrics at once

Real edge measurements move together. Push the device harder and utilization rises, power climbs with it, temperature follows, and throughput trades off against latency. Model that with a small **experiment**: sweep one knob, the model size, and record every metric across many trials.

**[Notebook cell]** Define the experiment. Three models from small to large, 30 trials each. Bigger models are slower, draw more power, and run hotter, so each gets its own baseline:

In [ ]:
models = {
    #          latency  power  utilization
    "yolov8n": dict(median=8.0,  baseWatts=18.0, util=45.0),
    "yolov8s": dict(median=14.0, baseWatts=26.0, util=68.0),
    "yolov8m": dict(median=26.0, baseWatts=38.0, util=88.0),
}
trialsPerModel = 30
print("models:", list(models))
print("rows to generate:", len(models) * trialsPerModel)

**[Notebook cell]** Generate the measurements. Reset the seed first so the experiment does not depend on the exploring you did above. Each trial draws a latency, then derives the metrics that depend on it: throughput is the inverse of latency, power rises with utilization plus noise, temperature climbs with power, and energy per inference is power times latency:

In [ ]:
rng = np.random.default_rng(SEED)   # reset: the experiment is independent of Parts 2-3

records = []
for modelName, cfg in models.items():
    lat = sampleLatencyMs(rng, trialsPerModel, median=cfg["median"])
    util = np.clip(cfg["util"] + rng.normal(0, 4, trialsPerModel), 0, 100)
    power = cfg["baseWatts"] + 0.12 * util + rng.normal(0, 1.0, trialsPerModel)
    temp = 40 + 0.6 * power + rng.normal(0, 1.5, trialsPerModel)
    for i in range(trialsPerModel):
        records.append({
            "model": modelName,
            "trial": i + 1,
            "latency_ms": round(float(lat[i]), 3),
            "throughput_fps": round(1000.0 / float(lat[i]), 2),
            "power_w": round(float(power[i]), 2),
            "gpu_util_pct": round(float(util[i]), 1),
            "temp_c": round(float(temp[i]), 1),
            "energy_mj": round(float(power[i]) * float(lat[i]), 1),
        })

print("generated", len(records), "measurements")
print("first row:", records[0])

---
## Part 5 · Record it in a tidy format

Data is only useful if the next tool can read it. The standard is **tidy** (or long) form: **one row per measurement**, **one column per variable**, and **units in the column names**. That is exactly what you built above, and it loads straight into pandas and every plotting tool.

**[Notebook cell]** Load the records into a `pandas` DataFrame and save it as CSV, the universal spreadsheet format:

In [ ]:
import pandas as pd

frame = pd.DataFrame(records)
csvPath = labDir / "experiment.csv"
frame.to_csv(csvPath, index=False)
print("wrote", csvPath)
frame.head()

**[Notebook cell]** Also save **JSON Lines**, one JSON object per line. It is the format the telemetry lab streams, and it is easy to append to as new measurements arrive:

In [ ]:
jsonlPath = labDir / "experiment.jsonl"
frame.to_json(jsonlPath, orient="records", lines=True)
print("wrote", jsonlPath)
print("first line:", jsonlPath.read_text().splitlines()[0])

**[Notebook cell]** Finally, write a small **schema** file describing the columns and their units. Future you, and anyone you share the data with, will know exactly what each number means:

In [ ]:
import json

schema = {
    "seed": SEED,
    "trials_per_model": trialsPerModel,
    "columns": {
        "model": "model name",
        "trial": "trial index, 1-based, after warm-up",
        "latency_ms": "inference latency, milliseconds",
        "throughput_fps": "frames per second, 1000/latency",
        "power_w": "board power draw, watts",
        "gpu_util_pct": "GPU utilization, percent",
        "temp_c": "package temperature, Celsius",
        "energy_mj": "energy per inference, millijoules",
    },
}
(labDir / "schema.json").write_text(json.dumps(schema, indent=2))
print("wrote", labDir / "schema.json")

In [ ]:
checkpoint("Part 5 - a tidy dataset on disk", [
    check("experiment.csv was written", fileNonEmpty("~/experimentLab/experiment.csv", minLines=90),
          hint="Run the to_csv cell. It should have one row per measurement plus a header."),
    check("experiment.jsonl is valid JSON Lines", jsonLinesValid("~/experimentLab/experiment.jsonl", requiredKeys=["model", "latency_ms", "power_w"], minRecords=90),
          hint="Run the to_json cell that writes experiment.jsonl."),
    check("schema.json documents the columns", fileContains("~/experimentLab/schema.json", "latency_ms"),
          hint="Run the schema cell."),
], successNote="One row per measurement, units in the names, a schema beside it. That is data other people can trust and reuse.")

---
## Part 6 · Make it reproducible: the classic way (venv and pip)

You have data, but you made it with whatever packages happen to be in this notebook. To let someone else reproduce it exactly, you package the code into a script and pin the packages. The traditional Python tools for that are **venv**, a private package folder, and **pip**, the installer.

The cells below use `!` so you can run them here, but this is ordinary terminal work. In a real terminal you would first `source classicenv/bin/activate`; from the notebook we just call the environment's tools by path, like `classicenv/bin/pip`.

**[Notebook cell]** Move the experiment out of the notebook and into a script so it can run on its own. `%%writefile` saves the cell to a file. The path is relative, so it lands in your `experimentLab` folder:

In [ ]:
%%writefile collect.py
# collect.py - reproduce the experiment from a fixed seed
import numpy as np
import pandas as pd

SEED = 494
rng = np.random.default_rng(SEED)

def sampleLatencyMs(rng, count, median=12.0, sigma=0.35):
    return np.exp(np.log(median) + sigma * rng.standard_normal(count))

models = {
    "yolov8n": dict(median=8.0,  baseWatts=18.0, util=45.0),
    "yolov8s": dict(median=14.0, baseWatts=26.0, util=68.0),
    "yolov8m": dict(median=26.0, baseWatts=38.0, util=88.0),
}
trialsPerModel = 30

records = []
for modelName, cfg in models.items():
    lat = sampleLatencyMs(rng, trialsPerModel, median=cfg["median"])
    util = np.clip(cfg["util"] + rng.normal(0, 4, trialsPerModel), 0, 100)
    power = cfg["baseWatts"] + 0.12 * util + rng.normal(0, 1.0, trialsPerModel)
    temp = 40 + 0.6 * power + rng.normal(0, 1.5, trialsPerModel)
    for i in range(trialsPerModel):
        records.append({
            "model": modelName, "trial": i + 1,
            "latency_ms": round(float(lat[i]), 3),
            "throughput_fps": round(1000.0 / float(lat[i]), 2),
            "power_w": round(float(power[i]), 2),
            "gpu_util_pct": round(float(util[i]), 1),
            "temp_c": round(float(temp[i]), 1),
            "energy_mj": round(float(power[i]) * float(lat[i]), 1),
        })

pd.DataFrame(records).to_csv("experiment_repro.csv", index=False)
print("wrote experiment_repro.csv with", len(records), "rows")

**[Notebook cell]** Create a private environment named `classicenv` and see that it starts almost empty. `python -m venv` builds it: a folder with its own Python and its own packages, isolated from everyone else on the machine:

In [ ]:
!python -m venv classicenv
!classicenv/bin/pip install --quiet --upgrade pip
!classicenv/bin/pip list

**[Notebook cell]** Install the two packages the script needs, then **pin** them with `pip freeze` into `requirements.txt`. Anyone who installs from that file gets the same versions you used:

In [ ]:
!classicenv/bin/pip install --quiet numpy pandas
!classicenv/bin/pip freeze > requirements.txt
print("pinned packages:")
print(Path("requirements.txt").read_text())

**[Notebook cell]** Run the script with the environment's Python. Because it uses those packages and the fixed seed, it reproduces the dataset exactly, into `experiment_repro.csv`:

In [ ]:
!classicenv/bin/python collect.py

In [ ]:
checkpoint("Part 6 - a pinned environment with venv and pip", [
    check("requirements.txt pins the packages", fileContains("~/experimentLab/requirements.txt", "numpy"),
          hint="Run the pip freeze cell."),
    check("the script reproduced the data", fileNonEmpty("~/experimentLab/experiment_repro.csv", minLines=90),
          hint="Run the collect.py cell with classicenv/bin/python."),
], successNote="venv isolates the packages and pip pins them. That is the classic reproducible setup, and you will meet it everywhere.")

---
## Part 7 · The same idea, faster: uv

`venv` plus `pip` plus `requirements.txt` is three tools and several steps. **uv** does all of it in one fast tool, and it is what this course's servers already use. `uv` creates the environment, resolves and installs packages, records exact versions in a lock file, and runs your script, all from one command.

**[Notebook cell]** The class image ships `uv`. If your environment does not have it yet, this installs it. Either way you end with a working `uv`:

In [ ]:
import shutil, sys
if shutil.which("uv") is None:
    !{sys.executable} -m pip install --quiet uv
!uv --version

**[Notebook cell]** Start a uv project and add the same two packages. `uv init` writes a `pyproject.toml`, the project's description, and `uv add` installs numpy and pandas into a fresh `.venv` while recording exact versions in `uv.lock`:

In [ ]:
!uv init --no-workspace --vcs none 2>/dev/null || true
!uv add numpy pandas
print("--- pyproject.toml ---")
print(Path("pyproject.toml").read_text())

**[Notebook cell]** Now run the script with `uv run`. It uses the project's locked environment automatically, with no activation step. You get the same reproduced dataset, from one command:

In [ ]:
!uv run collect.py

`uv.lock` now records the exact version of every package, including the dependencies of numpy and pandas. Commit `pyproject.toml` and `uv.lock` next to your code and anyone can run `uv run collect.py` and get the same data.

In [ ]:
checkpoint("Part 7 - the same, in one tool: uv", [
    check("uv wrote a lock file", fileExists("~/experimentLab/uv.lock"),
          hint="Run the uv add cell."),
    check("uv run reproduced the data", fileNonEmpty("~/experimentLab/experiment_repro.csv", minLines=90),
          hint="Run the uv run collect.py cell."),
], successNote="One tool, one lock file, one command to reproduce. This is why the course servers use uv.")

---
## Part 8 · Bonus: use your uv environment as a notebook kernel

Everything above ran scripts from the terminal. Sometimes you want your **notebook cells** to run inside a project environment instead, so the packages your cells use are the ones you pinned. You do that by registering the environment as a Jupyter **kernel**.

**[Notebook cell]** Add `ipykernel` to the uv project, then register it as a kernel named for your project. After this it appears in the kernel picker:

In [ ]:
!uv add --quiet ipykernel
!uv run python -m ipykernel install --user --name experimentlab --display-name "Python (experimentLab)"

Now open the kernel picker at the top right of the notebook, or use **Kernel > Change Kernel**, and choose **Python (experimentLab)**. Your cells then run in the uv environment. Switch back to **Python 3** to finish this lab.

You will not always need a project kernel, but when a lab depends on specific package versions, this keeps your notebook and your scripts using exactly the same ones.

In [ ]:
checkpoint("Part 8 - a project kernel", [
    check("the experimentlab kernel is registered", commandSucceeds("jupyter kernelspec list | grep -qi experimentlab"),
          hint="Run the ipykernel install cell."),
], successNote="Your uv environment is now selectable as a notebook kernel. Notebook and scripts, same packages.")

---
## Part 9 · Prove it reproduced, then sanity-check

Two final habits: confirm the reproduction really matches, and look the dataset over before you trust it.

**[Notebook cell]** Compare the dataset you made in the notebook with the one the script reproduced. Same seed, same code, so the numbers should match:

In [ ]:
import pandas as pd

original = pd.read_csv(labDir / "experiment.csv")
repro = pd.read_csv(labDir / "experiment_repro.csv")
numCols = original.select_dtypes("number").columns
print("rows:", len(original), "| numbers match reproduction:", np.allclose(original[numCols], repro[numCols]))

**[Notebook cell]** Finally, sanity-check the numbers. `describe()` summarizes each model, and a few asserts catch impossible values before they ever reach a plot:

In [ ]:
print(original.groupby("model")["latency_ms"].agg(["mean", "min", "max"]))
assert (original["latency_ms"] > 0).all(), "latency must be positive"
assert original["gpu_util_pct"].between(0, 100).all(), "utilization must be 0-100"
assert not original.isnull().any().any(), "there should be no missing values"
print("\nsanity checks passed")

In [ ]:
checkpoint("Part 9 - reproduced and sanity-checked", [
    check("a reproduction exists", fileNonEmpty("~/experimentLab/experiment_repro.csv", minLines=90),
          hint="Run Part 6 or Part 7 so the reproduction exists."),
    check("dataset is ready for the next lab", fileNonEmpty("~/experimentLab/experiment.csv", minLines=90),
          hint="Run Part 5 so experiment.csv exists."),
], successNote="You designed an experiment, chose a sample size, recorded tidy data, and made it reproducible. Next lab you turn it into figures.")

---
**[Notebook cell]** Optional cleanup. Uncomment to remove the environments, but keep your dataset for the next lab. Do not delete `experiment.csv`, the plotting lab uses it.

In [ ]:
# import shutil
# shutil.rmtree(labDir / "classicenv", ignore_errors=True)
# shutil.rmtree(labDir / ".venv", ignore_errors=True)
# print("removed environments; kept experiment.csv")

### Lab scorecard

In [ ]:
labSummary("Synthetic Experiment Data")

---

You now have a reproducible experiment and a tidy dataset in `~/experimentLab`. Keep it. In **Lab DD · Research-Quality Figures** you turn `experiment.csv` into plots you could put in a paper.

---
### One-minute feedback

Your feedback shapes the next version of this lab. Rate it, add anything that was confusing or broken, and click **Submit**. It takes about 30 seconds and goes straight to the instructor.

In [ ]:
feedback("Synthetic Experiment Data")